In [ ]:
from pyspark.sql.functions import count, sum, avg, max, min, round, to_date, col
from datetime import date
from dateutil.relativedelta import relativedelta

from nyctaxi_project.transformations.modules.utils.date_utils import get_month_start_n_months_ago

four_months_ago_start = get_month_start_n_months_ago(4)

df = spark.read.table("nyctaxi.02_silver.yellow_trips_enriched").filter(f"tpep_pickup_datetime > '{four_months_ago_start}'")

df = df.groupBy(to_date(col("tpep_pickup_datetime")).alias("pickup_date")).\
    agg(
        count("*").alias("total_trips"),
        round(avg("passenger_count"),1).alias("avg_passenger_per_trip"),
        round(avg("trip_distance"),1).alias("avg_distance_per_trip"),
        round(avg("fare_amount"),2).alias("avg_fare_per_trip"),
        max("fare_amount").alias("max_fare"),
        min("fare_amount").alias("min_fare"),
        round(sum("total_amount"),2).alias("total_revenue")
    )

df.display()

df.write.mode("append").saveAsTable("nyctaxi.03_gold.daily_trip_summary")

In [ ]:
spark.read.table("nyctaxi.03_gold.daily_trip_summary").display()